# MOMENT (frozen backbone + MIL head)

사전학습된 시계열 foundation model **MOMENT-1-base**(AutonLab)에 MIL classifier를 붙여, Holter ECG만으로 4년 내 SCD를 예측합니다.

- MOMENT backbone은 고정(사전학습 가중치 그대로), 새로 추가한 MIL head만 학습합니다.
- 입력: 3유도(X·Y·Z), 환자당 30초 구간 64개, 200 Hz → 100 Hz. 30초 구간마다 512 샘플(5.12초) 조각 5개를 유도별로 임베딩해 평균, 3유도는 이어 붙입니다.
- MIL head: 구간 임베딩을 합쳐 줄이고(channel fusion) → attention으로 64개 구간을 가중 평균 → SCD 확률
- head 설정: hidden 128, dropout 0.3, lr 1e-3, weight decay 0.01, batch 16, 최대 200 epoch (patience 30), best epoch 선택 기준 validation AP
- Z 유도가 없는 2유도 환자는 학습에서만 제외하고, 평가(validation/test)에는 X·Y 평균으로 Z를 채워 포함합니다.
- 저장: `Members/mingyu/results/moment/<run_id>/` (`training_history.csv`, `validation_predictions.csv`, `test_predictions.csv`, `metrics.json`, `metadata.json`, `head.pt`)
- 임베딩 캐시(`datas/moment_cache/`)가 없으면 처음에 다시 계산합니다 (GPU에서 시간이 걸림).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch

repo = Path.cwd().resolve()
while not (repo / "Members/mingyu/model/basic.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from Members.mingyu.model.config import load_basic_config  # noqa: E402
from Members.mingyu.model.finetuning import moment_probe as mp  # noqa: E402

## 실행 설정

In [ ]:
settings = {
    "model_name": "AutonLab/MOMENT-1-base",  # levidio/MOMENT-1-base는 이 모델의 복사본
    # 입력
    "n_windows": 64,  # 환자당 30초 window 수 (24시간에 고르게 분포)
    "decimation": 2,  # 200 Hz -> 100 Hz
    "chunks_per_window": 5,  # window당 512샘플 chunk 수 (100 Hz에서 최대 5)
    "embed_batch_size": 256,  # GPU 메모리 부족하면 줄이기
    # classifier head (새로 추가, 학습되는 부분)
    "hidden_dim": 128,
    "dropout": 0.3,
    # 학습
    "learning_rate": 1e-3,
    "weight_decay": 1e-2,
    "batch_size": 16,
    "epochs": 200,
    "patience": 30,
    "min_delta": 1e-4,
    "selection_metric": "val_ap",  # "val_ap" 또는 "val_loss"
    "target_recall": 0.75,  # validation에서 recall 75%가 되는 threshold 선택
    "seed": 42,
    # Z 없는 2유도 환자: "train" = 학습에서만 제외(평가는 포함, ML과 같은 환자로 비교)
    #                   "all" = 학습·평가 모두 제외, "none" = 모두 포함
    "exclude_two_lead": "train",
}
run_final_test = True  # PPT의 Prediction(test) 칸용

basic_path = repo / "Members/mingyu/model/basic.yaml"
basic = load_basic_config(basic_path)
raw = basic["data"]["raw_ecg"]
settings["sample_rate_hz"] = int(raw["sample_rate_hz"])
settings["segment_seconds"] = int(raw["segment_seconds"])
settings["target_semantics"] = raw["target_semantics"]
tag = f"{settings['model_name'].split('/')[-1].lower()}_w{settings['n_windows']}_d{settings['decimation']}_c{settings['chunks_per_window']}"
settings["cache_dir"] = str(repo / "Members/mingyu/datas/moment_cache" / tag)
output_root = repo / "Members/mingyu/results/moment"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mp.seed_everything(settings["seed"])
print(device, settings["cache_dir"])

## 1. 코호트

In [ ]:
cohort = mp.load_cohort(
    Path(raw["split_path"]),
    Path(raw["quality_index_path"]),
    Path(raw["wave_directory"]),
    Path(raw["lead_metadata_path"]),
)
print(cohort.groupby("split").y_true.agg(patients="count", scd="sum"))
print("Z 없음(XY만):", (~cohort.has_z).sum(), "개 파일 — 빠진 Z는 X·Y 평균 임베딩으로 대체")

## 2. MOMENT 불러오기 + freeze 확인

`trainable`이 head 파라미터 수와 같고 backbone은 전부 `frozen`이어야 합니다.

In [ ]:
backbone = mp.load_moment(settings["model_name"], device)
d_model = (
    backbone.config.d_model
    if hasattr(backbone, "config") and hasattr(backbone.config, "d_model")
    else 768
)
n_leads = 3
head = mp.MILClassifierHead(n_leads * d_model, settings["hidden_dim"], settings["dropout"])
model = mp.MomentSCDClassifier(backbone, head)
print("backbone:", mp.count_parameters(model.backbone))
print("head    :", mp.count_parameters(model.head))
print("total   :", mp.count_parameters(model))

## 3. Window 임베딩 추출 (frozen backbone, 1회)

캐시가 이미 있으면 건너뜁니다. 설정(`n_windows`, `decimation`, `chunks_per_window`, 모델)을 바꾸면 새 캐시 폴더가 생깁니다. 중간에 멈춰도 다시 실행하면 이어서 진행합니다.

In [ ]:
cache_dir = mp.extract_embeddings(cohort, backbone, settings, device)
backbone.cpu()
torch.cuda.empty_cache()

## 4. Classifier head 학습 (새 레이어만)

In [ ]:
excluded_splits = {"none": (), "train": ("train",), "all": ("train", "validation", "test")}[
    settings["exclude_two_lead"]
]
parts = {}
for name in ("train", "validation", "test"):
    frame = cohort.loc[cohort.split.eq(name)]
    if name in excluded_splits:
        frame = frame.loc[frame.has_z]
    parts[name] = frame.reset_index(drop=True)
print(
    pd.DataFrame(
        {
            name: {
                "patients": len(f),
                "scd": int(f.y_true.sum()),
                "two_lead": int((~f.has_z).sum()),
            }
            for name, f in parts.items()
        }
    )
)

bags = {name: mp.load_bags(frame, cache_dir) for name, frame in parts.items()}
labels = {name: frame.y_true.to_numpy() for name, frame in parts.items()}

mp.seed_everything(settings["seed"])
head = mp.MILClassifierHead(bags["train"][0].shape[-1], settings["hidden_dim"], settings["dropout"])
best_row, history = mp.train_head(
    head, bags["train"], labels["train"], bags["validation"], labels["validation"], settings, device
)
print("best epoch:", best_row)
history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(8, 3))
history.plot(x="epoch", y=["val_auroc", "val_ap"], figsize=(8, 3))

## 5. Validation 평가 + 저장

In [ ]:
run_id, run_dir = mp.make_run_dir(output_root, settings["model_name"])

val_p = mp.predict(head, *bags["validation"], device)
threshold = mp.recall_threshold(labels["validation"], val_p, settings["target_recall"])
val_metrics, val_frame = mp.evaluate(parts["validation"], val_p, threshold, run_id)

torch.save(head.state_dict(), run_dir / "head.pt")
history.to_csv(run_dir / "training_history.csv", index=False)
val_frame.to_csv(run_dir / "validation_predictions.csv", index=False)
mp.save_json(run_dir / "metrics.json", {"validation": val_metrics, "best_epoch": best_row})
mp.save_json(
    run_dir / "metadata.json",
    {
        "model": "MOMENT+MIL",
        "run_id": run_id,
        "backbone": settings["model_name"],
        "backbone_frozen": True,
        "target_column": "SCD",
        "class_mapping": {"negative": 0, "positive": 1},
        "classification_threshold": threshold,
        "settings": settings,
        "counts": {
            name: {"patients": len(f), "scd": int(f.y_true.sum())} for name, f in parts.items()
        },
    },
)
print(run_dir)
pd.Series(val_metrics)

## 6. 최종 test (모델 선택이 끝난 뒤 `run_final_test=True`)

In [ ]:
if run_final_test:
    test_p = mp.predict(head, *bags["test"], device)
    test_metrics, test_frame = mp.evaluate(parts["test"], test_p, threshold, run_id)
    test_frame.to_csv(run_dir / "test_predictions.csv", index=False)
    mp.save_json(
        run_dir / "metrics.json",
        {"validation": val_metrics, "test": test_metrics, "best_epoch": best_row},
    )
    display(pd.Series(test_metrics))
else:
    print("test 건너뜀")